In [1]:
import ctypes
import math
import numpy as np
import os
import sys
import time
import pydicom
import pandas as pd
from cuda import cuda, cudart
from py_cuda_helper import checkCudaErrors, findCudaDevice, KernelHelper
import plotly.express as px
import plotly.graph_objects as go

In [2]:
# import math
# import warp as wp
# import warp.render
# import warp.sim
# import warp.sim.render
# from pxr import Usd, UsdGeom, Sdf, Gf

In [3]:
devID = findCudaDevice()
deviceProps = checkCudaErrors(cudart.cudaGetDeviceProperties(devID))
print("CUDA device [{}] has {} Multi-Processors SM {}.{}".format(deviceProps.name,
                                                                     deviceProps.multiProcessorCount,
                                                                     deviceProps.major,
                                                                     deviceProps.minor))
if (deviceProps.major < 2):
    print("Requires SM 2.0 or higher for support of Texture Arrays.  Test will exit...")
    sys.exit()

CUDA device [b'NVIDIA GeForce RTX 4090'] has 128 Multi-Processors SM 8.9


In [4]:
deviceProps.maxThreadsPerBlock

1024

In [5]:
def get_rts_as_list(rtsfile):
    if rtsfile != "":
        ds = pydicom.dcmread(rtsfile)
        structures = []
        for item in ds.StructureSetROISequence:
            structures.append(item.ROIName)
        return structures
    else:
        return []

def get_structure_as_dataframe(rtsfile, roi_name, roi_id):
    if rtsfile.endswith(".dcm"):
        ds = pydicom.dcmread(rtsfile)
        #sinfo = ds.StructureSetROISequence[roi_id]
        cntr_sequence = ds.ROIContourSequence[roi_id].ContourSequence
        count = 0
        rows = []
        for item in cntr_sequence:
            sliceid = count
            points = np.array(item.ContourData)
            pt_array = points.reshape(item.NumberOfContourPoints, 3)
            for i in range(0,item.NumberOfContourPoints):
                rows.append([roi_id, roi_name, sliceid, i, pt_array[i,0], pt_array[i,1], pt_array[i,2]])
            count += 1
    cntr = pd.DataFrame(rows,columns=['roi_id','roi_name','seq_id','pt_id','xpos','ypos','zpos'])
    return cntr

def get_rti_as_nparray(path):
    rtifiles = []
    for s in os.listdir(path):
        rtipath = os.path.join(path, s)
        rtifiles.append(rtipath)
    slices = []
    for file in rtifiles:
        ds = pydicom.dcmread(file)
        if hasattr(ds, 'SliceLocation'):
            slices.append(ds)
    
    slices = sorted(slices, key=lambda s: s.SliceLocation)
    image_size = (int(slices[0].Rows), int(slices[0].Columns), len(slices))
    voxel_size = (float(slices[0].PixelSpacing[0]), float(slices[0].PixelSpacing[1]), float(slices[0].SliceThickness))
    image_origin = slices[0].ImagePositionPatient

    uids = []
    pixels = np.zeros(image_size)
    for i, s in enumerate(slices):
        pixels[:,:,i] = s.pixel_array
        uids.append(s.SOPInstanceUID)

    return pixels, uids, image_origin, voxel_size

def get_rti_info(path):
    rtifiles = []
    for s in os.listdir(path):
        rtipath = os.path.join(path, s)
        rtifiles.append(rtipath)
    slices = []
    scount = 0
    smin = 0
    smax = 0
    for file in rtifiles:
        ds = pydicom.dcmread(file)
        if hasattr(ds, 'SliceLocation'):
            sliceloc = float(ds.SliceLocation)
            IOP = ds.ImageOrientationPatient
            IPP = ds.ImagePositionPatient
            normal = np.cross(IOP[0:3], IOP[3:])
            projection = np.dot(IPP, normal)
            slices += [{"p":projection, "ds":ds}]
            if scount == 0:
                smin = sliceloc
                smax = sliceloc
            else:
                if sliceloc < smin:
                    smin = sliceloc
                if sliceloc > smax:
                    smax = sliceloc
            scount += 1
            # 
            #     slices
    
    sorted_slices = sorted(slices, key=lambda s: -1*s["p"])
    image_size = [int(sorted_slices[0]["ds"].Rows), int(sorted_slices[0]["ds"].Columns), len(sorted_slices)]
    voxel_size = [float(sorted_slices[0]["ds"].PixelSpacing[0]), float(sorted_slices[0]["ds"].PixelSpacing[1]), float(sorted_slices[0]["ds"].SliceThickness)]
    image_origin = slices[0]["ds"].ImagePositionPatient
    image_orient = slices[0]["ds"].ImageOrientationPatient

    return image_origin, voxel_size, image_size, image_orient, [smin, smax]

In [6]:
rtspath = "/media/jacko/Samsung_T5/NCI_Phantoms/ReferenceDataSets/35m_noa_icrp_267x194x796/nci_30m_noa_icrp_267x194x796_rtstruct.dcm"
rtipath = "/media/jacko/Samsung_T5/NCI_Phantoms/ReferenceDataSets/35m_noa_icrp_267x194x796/"

In [7]:
img_origin, img_resolution, img_size, img_orient, img_extent = get_rti_info(rtipath)
print("Image Dimensions: ", img_size)
print("Image Voxel Size: ", img_resolution)
print("Image Origin: ", img_origin)
print("Image Orientation: ", img_orient)
print("Image Extent: ", img_extent)

Image Dimensions:  [194, 267, 796]
Image Voxel Size:  [1.579, 1.579, 2.207]
Image Origin:  [0.7895, 0.7895, 1755.6685]
Image Orientation:  [1, 0, 0, 0, 1, 0]
Image Extent:  [1.1035, 1755.6685]


In [8]:
rois = get_rts_as_list(rtspath)
rois

['Brain',
 'Pituitary',
 'SpinalCord',
 'Eye_L',
 'Eye_R',
 'Eyes',
 'Lens_L',
 'Lens_R',
 'Lens',
 'Nose',
 'Ear_Externals',
 'Tongue',
 'Tonsil',
 'Parotids',
 'Esophagus',
 'Bronchus',
 'Trachea',
 'Larynx',
 'Heart',
 'Lung_L',
 'Lung_R',
 'Lung_Tot',
 'Liver',
 'Kidney_L',
 'Kidney_R',
 'Kidney_Tot',
 'Colon_Ascending',
 'Colon_Sigmoid',
 'Colon_Descending',
 'Colon_Tot',
 'Gallbladder',
 'Stomach',
 'Bowel_Small',
 'Pancreas',
 'Spleen',
 'Bladder',
 'Glnd_Adrenal_L',
 'Glnd_Adrenal_R',
 'Glnd_Adrenal_Tot',
 'Glnd_Thymus',
 'Glnd_Thyroid',
 'Glnd_Submands',
 'Glnd_Sublngs',
 'Breasts',
 'Penis',
 'Scrotum',
 'Testes',
 'Scrotum_Tot',
 'Penis_Tot',
 'Breast_Adipose',
 'Prostate',
 'Cranium',
 'Mandible',
 'Scapulae',
 'Clavicles',
 'Sternum',
 'Ribs',
 'Vertebrae_C',
 'Vertebrae_T',
 'Vertebrae_L',
 'Sacrum',
 'Os_coxae',
 'Femora_upper',
 'Femora_lower',
 'Tibiae_fibiae_patellae',
 'Ankle_foot',
 'Humeri_upper',
 'EXTERNAL']

In [ ]:
roi_name = input()

In [ ]:
roi_id = rois.index(roi_name)

In [ ]:
roidf = get_structure_as_dataframe(rtspath, roi_name, roi_id)
print("ROI Lateral Extent: ", roidf['xpos'].min(), roidf['xpos'].max())
print("ROI AP Extent: ", roidf['ypos'].min(), roidf['ypos'].max())
print("ROI SI Extent: ", roidf['zpos'].min(), roidf['zpos'].max())
roidf.head()

ROI Lateral Extent:  241.98175 249.08725
ROI AP Extent:  82.50275 86.45025
ROI SI Extent:  1643.1115 1649.7325


,roi_id,roi_name,seq_id,pt_id,xpos,ypos,zpos
0,6,Lens_L,0,0,241.98175,84.08175,1649.7325
1,6,Lens_L,0,1,242.77125,84.08175,1649.7325
2,6,Lens_L,0,2,243.56075,83.29225,1649.7325
3,6,Lens_L,0,3,243.56075,82.50275,1649.7325
4,6,Lens_L,0,4,247.50825,82.50275,1649.7325


In [ ]:
# fig = px.line_3d(roidf, x='xpos', y='ypos', z='zpos', color='seq_id')
# fig.show()

In [ ]:
CTR_RES = 0.2

In [ ]:
cntr_resolution = [0,0,0]
cntr_resolution[0] = img_resolution[0] * CTR_RES
cntr_resolution[1] = img_resolution[1] * CTR_RES
cntr_resolution[2] = img_resolution[2]
print(cntr_resolution)

roidf['x_ijk'] = roidf['xpos'].apply(lambda i: int((i - img_origin[0]) / cntr_resolution[0]))
roidf['y_ijk'] = roidf['ypos'].apply(lambda i: int((i - img_origin[1]) / cntr_resolution[1]))
roidf['z_ijk'] = roidf['zpos'].apply(lambda i: int((i - img_extent[0]) / cntr_resolution[2]))

print("ROI Lateral Extent: ", roidf['x_ijk'].min(), roidf['x_ijk'].max())
print("ROI AP Extent: ", roidf['y_ijk'].min(), roidf['y_ijk'].max())
print("ROI SI Extent: ", roidf['z_ijk'].min(), roidf['z_ijk'].max())

[0.3158, 0.3158, 2.207]
ROI Lateral Extent:  763 786
ROI AP Extent:  258 271
ROI SI Extent:  744 747


In [ ]:
cntr_buffer = [20,20,5]

mins = [roidf['x_ijk'].min(), roidf['y_ijk'].min(), roidf['z_ijk'].min()]
maxs = [roidf['x_ijk'].max(), roidf['y_ijk'].max(), roidf['z_ijk'].max()]
print(mins)
print(maxs)

cntr_arraySize = [maxs[0] - mins[0] + 40, maxs[1] - mins[1] + 40, maxs[2] - mins[2] + 10]
print(cntr_arraySize)


[763, 258, 744]
[786, 271, 747]
[63, 53, 13]


In [ ]:
roidf['cntr_x_ijk'] = roidf['x_ijk'].apply(lambda i: int(i - mins[0] + cntr_buffer[0]))
roidf['cntr_y_ijk'] = roidf['y_ijk'].apply(lambda i: int(i - mins[1] + cntr_buffer[1]))
roidf['cntr_z_ijk'] = roidf['z_ijk'].apply(lambda i: int(i - mins[2] + cntr_buffer[2]))
roidf.head()

,roi_id,roi_name,seq_id,pt_id,xpos,ypos,zpos,x_ijk,y_ijk,z_ijk,cntr_x_ijk,cntr_y_ijk,cntr_z_ijk
0,6,Lens_L,0,0,241.98175,84.08175,1649.7325,763,263,747,20,25,8
1,6,Lens_L,0,1,242.77125,84.08175,1649.7325,766,263,747,23,25,8
2,6,Lens_L,0,2,243.56075,83.29225,1649.7325,768,261,747,25,23,8
3,6,Lens_L,0,3,243.56075,82.50275,1649.7325,768,258,747,25,20,8
4,6,Lens_L,0,4,247.50825,82.50275,1649.7325,781,258,747,38,20,8


In [ ]:
cntr_volume = np.zeros(cntr_arraySize[0]*cntr_arraySize[1]*cntr_arraySize[2], dtype=np.float32)

In [ ]:
for i, row in roidf.iterrows():
	index = row['cntr_x_ijk'] + cntr_arraySize[0]*row['cntr_y_ijk'] + cntr_arraySize[0]*cntr_arraySize[1]*row['cntr_z_ijk']
	cntr_volume[index] = 1.0

In [ ]:
c_img = cntr_volume.reshape(cntr_arraySize[2],cntr_arraySize[1],cntr_arraySize[0]).T
px.imshow(c_img[:,:,int(0.5*cntr_arraySize[2])])

In [ ]:
def modifiedBresenham(df, volume, arraySize, value = 1.0):
	clist = df['seq_id'].unique()
	for _, seq in enumerate(clist):
		newslice = True
		last = [0,0,0]
		cdf = df[df['seq_id'] == seq]
		for p in range(len(cdf.index)+1):
			if (p == len(cdf.index)):
				coord = [ cdf.iloc[0]['cntr_x_ijk'], cdf.iloc[0]['cntr_y_ijk'], cdf.iloc[0]['cntr_z_ijk'] ]
			else:
				coord = [ cdf.iloc[p]['cntr_x_ijk'], cdf.iloc[p]['cntr_y_ijk'], cdf.iloc[p]['cntr_z_ijk'] ]

			if newslice:
				newslice = False
			else:
				diff = [coord[0] - last[0], coord[1] - last[1]]
				dir = [1,1]
				if diff[0] < 0: dir[0] = -1
				if diff[1] < 0: dir[1] = -1
				stepSwitch = abs(diff[0]) > abs(diff[1])
				if (diff[1] == 0):
					slope = [1,0]
				elif (diff[0] == 0):
					slope = [0,1]
				elif (stepSwitch):
					slope = [abs(diff[0]) / abs(diff[1]) , 1]
				else:
					slope = [1, abs(diff[1]) / abs(diff[0])]

				step = [last[0], last[1]]
				subStep = [0,0]

				while (step[0] != coord[0] or step[1] != coord[1]):
					if (stepSwitch):
						if (step[0] != coord[0]):
							step[0] += dir[0]
							subStep[0] += 1
							index = step[0] + arraySize[0]*step[1] + arraySize[0]*arraySize[1]*coord[2]
							volume[index] = value

							if (subStep[0] == slope[0]):
								stepSwitch = not stepSwitch
								subStep[0] = 0
						else:
							stepSwitch = not stepSwitch
					else:
						if (step[1] != coord[1]):
							step[1] += dir[1]
							subStep[1] += 1
							index = step[0] + arraySize[0]*step[1] + arraySize[0]*arraySize[1]*coord[2]
							volume[index] = value

							if (subStep[1] == slope[1]):
								stepSwitch = not stepSwitch
								subStep[1] = 0
						else:
							stepSwitch = not stepSwitch
			
			last[0] = coord[0]
			last[1] = coord[1]
			last[2] = coord[2]




In [ ]:
modifiedBresenham(roidf, cntr_volume, cntr_arraySize)

In [ ]:
c_img = cntr_volume.reshape(cntr_arraySize[2],cntr_arraySize[1],cntr_arraySize[0]).T
px.imshow(c_img[:,:,int(0.5*cntr_arraySize[2])])

In [ ]:
# send the contour volume, containing the subcontour outlines, to texture memory on the GPU
# /////////////////// Bind Inputs to 3D Texture Arrays //////////////////////////////////////////////
cExtent = cudart.make_cudaExtent(cntr_arraySize[0], cntr_arraySize[1], cntr_arraySize[2])
channelDesc	= checkCudaErrors(cudart.cudaCreateChannelDesc(32, 0, 0, 0, cudart.cudaChannelFormatKind.cudaChannelFormatKindFloat))
cu_3darray =  checkCudaErrors(cudart.cudaMalloc3DArray(channelDesc, cExtent, cudart.cudaArrayDefault))

copyParams = cudart.cudaMemcpy3DParms()
copyParams.srcPos 		= 	cudart.make_cudaPos(0,0,0)
copyParams.dstPos 		= 	cudart.make_cudaPos(0,0,0)
copyParams.srcPtr 		=	cudart.make_cudaPitchedPtr(cntr_volume, cExtent.width*np.dtype(np.float32).itemsize, cExtent.width, cExtent.height);
copyParams.dstArray		= 	cu_3darray
copyParams.extent       =	cExtent
copyParams.kind		    =	cudart.cudaMemcpyKind.cudaMemcpyHostToDevice
checkCudaErrors(cudart.cudaMemcpy3D(copyParams))

In [ ]:
texRes = cudart.cudaResourceDesc()
texRes.resType = cudart.cudaResourceType.cudaResourceTypeArray
texRes.res.array.array = cu_3darray

texCntr = cudart.cudaTextureDesc()
texCntr.normalizedCoords	=	False
texCntr.filterMode			=	cudart.cudaTextureFilterMode.cudaFilterModeLinear
texCntr.addressMode[0]		=	cudart.cudaTextureAddressMode.cudaAddressModeClamp
texCntr.addressMode[1]		=	cudart.cudaTextureAddressMode.cudaAddressModeClamp
texCntr.addressMode[2]		=	cudart.cudaTextureAddressMode.cudaAddressModeClamp
texCntr.readMode = cudart.cudaTextureReadMode.cudaReadModeElementType

tex = checkCudaErrors(cudart.cudaCreateTextureObject(texRes, texCntr, None))

In [ ]:
dvol_size = cntr_arraySize[0] * cntr_arraySize[1] * cntr_arraySize[2] * np.dtype(np.float32).itemsize
d_vol = checkCudaErrors(cudart.cudaMalloc(dvol_size))

In [ ]:
checkCudaErrors(cudart.cudaMemset(d_vol, 0, dvol_size))

In [ ]:
dimBlock = cudart.dim3()
dimBlock.x = 32
dimBlock.y = 1
dimBlock.z = 1

gridx = cntr_arraySize[0]/dimBlock.x
if ( cntr_arraySize[0] % dimBlock.x > 0): gridx += 1
gridy = cntr_arraySize[1]/dimBlock.x
if ( cntr_arraySize[1] % dimBlock.x > 0): gridy += 1
gridz = cntr_arraySize[2]
	
dimGridx = cudart.dim3()
dimGridx.x = gridx
dimGridx.y = gridz
dimGridx.z = 1

dimGridy = cudart.dim3()
dimGridy.x = gridy
dimGridy.y = gridz
dimGridy.z = 1

print(dimBlock)
print(dimGridx)
print(dimGridy)

x : 32
y : 1
z : 1
x : 2
y : 13
z : 1
x : 2
y : 13
z : 1


In [ ]:
# contourFiller = '''
# extern "C" __global__ void
# cudaIndexingCheck(float *include, int d_sizex, int d_sizey, int d_sizez, float *source)
# {
#     unsigned int X = threadIdx.x + blockDim.x*blockIdx.x;
#     if (X >= d_sizex) return;
#     unsigned int Z = threadIdx.y + blockDim.y*blockIdx.y;
#     if (Z >= d_sizez) return;
	
# 	for (unsigned int Y = 0; Y<d_sizey; Y++) {
#         int index = X + Y * d_sizex + Z * d_sizex * d_sizey;
# 		include[index] = index;
# 	}
# }
# '''

In [ ]:
contourFiller = '''
extern "C" __global__ void
cudaContourFillerX( float *include, int dir, int d_sizex, int d_sizey, int d_sizez, cudaTextureObject_t tex)
{
    unsigned int X = threadIdx.x + blockDim.x*blockIdx.x;
    if (X >= d_sizex) return;
    unsigned int Z = threadIdx.y + blockDim.y*blockIdx.y;
    if (Z >= d_sizez) return;

    float targetx, targety, targetz;
    targetx = __int2float_rn(X);
    targetz = __int2float_rn(Z);

    int spot = 0, last = -5;
    bool inc = 0;
    for (unsigned int Y = 0; Y<d_sizey; Y++) {

        int y = Y*dir + (1 - dir)*(d_sizey - 1 - Y);
        targety = __int2float_rn(y);

        int idx = __float2int_rn(targetx) +
                    d_sizex * ( __float2int_rn(targety) +
                      d_sizey * __float2int_rn(targetz) );

        spot = tex3D<float>(tex, targetx+0.5f, targety+0.5f, targetz+0.5f);
		
        if (spot > 0){
			int diff = Y - last;
            if (diff > 1)
                inc = !inc;
            last = Y;
        }

        if (inc)
            include[idx] += 2.0f;
    }
}

extern "C" __global__ void
cudaContourFillerY( float *include, int dir, int d_sizex, int d_sizey, int d_sizez, cudaTextureObject_t tex)
{
    int Y = threadIdx.x + blockDim.x*blockIdx.x;
    if (Y >= d_sizey) return;
    int Z = threadIdx.y + blockDim.y*blockIdx.y;
    if (Z >= d_sizez) return;

    float targetx, targety, targetz;
    targety = __int2float_rn(Y);
    targetz = __int2float_rn(Z);

    int spot = 0, last = -5;
    bool inc = 0;
    for (int X = 0; X<d_sizex; X++) {

        int x = X*dir + (1 - dir)*(d_sizex - 1 - X);
        targetx = __int2float_rn(x);

        int idx = __float2int_rn(targetx) +
                    d_sizex * ( __float2int_rn(targety) +
                      d_sizey * __float2int_rn(targetz) );

        spot =  __float2int_rn( tex3D<float>(tex, targetx+0.5f, targety+0.5f, targetz+0.5f) );
		
        if (spot > 0){
            int diff = X - last;
            if (diff > 1)
                inc = !inc;
            last = X;
        }

        if (inc)
            include[idx] += 2;
    }
}

extern "C" __global__ void
cudaContourFillerXdiag( float *include, int starty, int dirx, int d_sizex, int d_sizey, int d_sizez, cudaTextureObject_t tex)
{
    int X = threadIdx.x + blockDim.x*blockIdx.x;
    if (X >= d_sizex) return;
    int Y = starty * (d_sizey - 1);
    int Z = threadIdx.y + blockDim.y*blockIdx.y;
    if (Z >= d_sizez) return;

    float targetx, targety, targetz;
    targetz = __int2float_rn(Z);

    int spot = 0, lastx = -5, lasty = -5;
    int diry = 1;
    if (starty == 1) diry = -1;
    bool inc = 0;
    for (int c=0; c<d_sizex*2; c++)
    {
        if (c%2==0)
            X+=dirx;
        else
            Y+=diry;

        if (X < 0 || Y < 0 || X >= d_sizex || Y >= d_sizey ) return;

        targetx = __int2float_rn(X);
        targety = __int2float_rn(Y);

        int idx = __float2int_rn(targetx) +
                    d_sizex * ( __float2int_rn(targety) +
                      d_sizey * __float2int_rn(targetz) );

        spot =  __float2int_rn( tex3D<float>(tex, targetx+0.5f, targety+0.5f, targetz+0.5f) );
        if (spot > 0){
            int diffx = (X - lastx);
            int diffy = (Y - lasty);
            if (diffx > 0 && diffy > 0)
                inc = !inc;
            lastx = X;
            lasty = Y;
            }

        if (inc)
            include[idx] += 1;
    }
}

extern "C" __global__ void
cudaContourFillerYdiag( float *include, int startx, int diry, int d_sizex, int d_sizey, int d_sizez, cudaTextureObject_t tex)
{
    int X = startx * (d_sizex - 1);
    int Y = 1 + threadIdx.x + blockDim.x*blockIdx.x;
    if (Y >= d_sizey - 2) return;
    int Z = threadIdx.y + blockDim.y*blockIdx.y;
    if (Z >= d_sizez) return;

	float3 checkings = make_float3(0,0,0);
    float targetx, targety, targetz;
    targetz = __int2float_rn(Z);

    int spot = 0, lastx = -5, lasty = -5;
    int dirx = 1;
    if (startx == 1) dirx = -1;
    bool inc = 0;
    for (int c=0; c<d_sizey*2; c++)
    {
        if (c%2==0)
            X+=dirx;
        else
            Y+=diry;

        if (X < 0 || Y < 0 || X >= d_sizex || Y >= d_sizey ) return;

        targetx = __int2float_rn(X);
        targety = __int2float_rn(Y);

        int idx = __float2int_rn(targetx) +
                    d_sizex * ( __float2int_rn(targety) +
                      d_sizey * __float2int_rn(targetz) );

        spot =  __float2int_rn( tex3D<float>(tex, targetx+0.5f, targety+0.5f, targetz+0.5f) );
        if (spot > 0){
            int diffx = (X - lastx);
            int diffy = (Y - lasty);
            if (diffx > 0 && diffy > 0)
                inc = !inc;
            lastx = X;
            lasty = Y;
            }

        if (inc)
            include[idx] += 1;
    }
}

extern "C" __global__ void
cudaZeroFillerIsotropic( float *include, int d_sizex, int d_sizey, int d_sizez, float ratiox, float ratioy, float ratioz, float threshold, cudaTextureObject_t tex)
{
    int X = threadIdx.x + blockDim.x*blockIdx.x;
    if (X >= d_sizex) return;
    int Y = threadIdx.y + blockDim.y*blockIdx.y;
    if (Y >= d_sizey) return;
    int Z = blockIdx.z;
    if (Z >= d_sizez) return;

    float targetx, targety, targetz;
    targetx = __int2float_rn(X) * ratiox;
    targety = __int2float_rn(Y) * ratioy;
    targetz = __int2float_rn(Z) * ratioz;
	
	int idx = X + d_sizex * Y + d_sizex * d_sizey * Z;
	
	float spot = tex3D<float>(tex, targetx+0.5f, targety+0.5f, targetz+0.5f);
	
	if (spot > threshold) {
        include[idx] = 1.0;
    }
}
'''

In [ ]:
kernelHelper = KernelHelper(contourFiller, devID)
# _cudaIndexingCheck = kernelHelper.getFunction(b'cudaIndexingCheck')
_cudaContourFillerX = kernelHelper.getFunction(b'cudaContourFillerX')
_cudaContourFillerY = kernelHelper.getFunction(b'cudaContourFillerY')
_cudaContourFillerXdiag = kernelHelper.getFunction(b'cudaContourFillerXdiag')
_cudaContourFillerYdiag = kernelHelper.getFunction(b'cudaContourFillerYdiag')
_cudaZeroFillerIsotropic = kernelHelper.getFunction(b'cudaZeroFillerIsotropic')

In [ ]:
# kernelArgs = ((d_vol, cntr_arraySize[0], cntr_arraySize[1], cntr_arraySize[2], d_src), (ctypes.c_void_p, ctypes.c_int, ctypes.c_int, ctypes.c_int, ctypes.c_void_p))
# checkCudaErrors(cuda.cuLaunchKernel(_cudaIndexingCheck,
# 									dimGridx.x, dimGridx.y, dimGridx.z,         # grid dim
# 									dimBlock.x, dimBlock.y, dimBlock.z,      # block dim
# 									0, 0,                                    # shared mem and stream
# 									kernelArgs, 0))   
# checkCudaErrors(cudart.cudaDeviceSynchronize())

In [ ]:
kernelArgs = ((d_vol, 0, cntr_arraySize[0], cntr_arraySize[1], cntr_arraySize[2], tex), (ctypes.c_void_p, ctypes.c_int, ctypes.c_int, ctypes.c_int, ctypes.c_int, None))
checkCudaErrors(cuda.cuLaunchKernel(_cudaContourFillerX,
									dimGridx.x, dimGridx.y, dimGridx.z,         # grid dim
									dimBlock.x, dimBlock.y, dimBlock.z,      # block dim
									0, 0,                                    # shared mem and stream
									kernelArgs, 0))   
checkCudaErrors(cudart.cudaDeviceSynchronize())
checkCudaErrors(cuda.cuLaunchKernel(_cudaContourFillerY,
									dimGridy.x, dimGridy.y, dimGridy.z,         # grid dim
									dimBlock.x, dimBlock.y, dimBlock.z,      # block dim
									0, 0,                                    # shared mem and stream
									kernelArgs, 0))   
checkCudaErrors(cudart.cudaDeviceSynchronize())

kernelArgs = ((d_vol, 1, cntr_arraySize[0], cntr_arraySize[1], cntr_arraySize[2], tex), (ctypes.c_void_p, ctypes.c_int, ctypes.c_int, ctypes.c_int, ctypes.c_int, None))
checkCudaErrors(cuda.cuLaunchKernel(_cudaContourFillerX,
									dimGridx.x, dimGridx.y, dimGridx.z,         # grid dim
									dimBlock.x, dimBlock.y, dimBlock.z,      # block dim
									0, 0,                                    # shared mem and stream
									kernelArgs, 0))   
checkCudaErrors(cudart.cudaDeviceSynchronize())
checkCudaErrors(cuda.cuLaunchKernel(_cudaContourFillerY,
									dimGridy.x, dimGridy.y, dimGridy.z,         # grid dim
									dimBlock.x, dimBlock.y, dimBlock.z,      # block dim
									0, 0,                                    # shared mem and stream
									kernelArgs, 0))   
checkCudaErrors(cudart.cudaDeviceSynchronize())


In [ ]:
kernelArgs = ((d_vol, 0, 1, cntr_arraySize[0], cntr_arraySize[1], cntr_arraySize[2], tex), (ctypes.c_void_p, ctypes.c_int, ctypes.c_int, ctypes.c_int, ctypes.c_int, ctypes.c_int, None))
checkCudaErrors(cuda.cuLaunchKernel(_cudaContourFillerXdiag,
									dimGridx.x, dimGridx.y, dimGridx.z,         # grid dim
									dimBlock.x, dimBlock.y, dimBlock.z,      # block dim
									0, 0,                                    # shared mem and stream
									kernelArgs, 0))   
checkCudaErrors(cudart.cudaDeviceSynchronize())
checkCudaErrors(cuda.cuLaunchKernel(_cudaContourFillerYdiag,
									dimGridy.x, dimGridy.y, dimGridy.z,         # grid dim
									dimBlock.x, dimBlock.y, dimBlock.z,      # block dim
									0, 0,                                    # shared mem and stream
									kernelArgs, 0))  
checkCudaErrors(cudart.cudaDeviceSynchronize())

kernelArgs = ((d_vol, 0, -1, cntr_arraySize[0], cntr_arraySize[1], cntr_arraySize[2], tex), (ctypes.c_void_p, ctypes.c_int, ctypes.c_int, ctypes.c_int, ctypes.c_int, ctypes.c_int, None))
checkCudaErrors(cuda.cuLaunchKernel(_cudaContourFillerXdiag,
									dimGridx.x, dimGridx.y, dimGridx.z,         # grid dim
									dimBlock.x, dimBlock.y, dimBlock.z,      # block dim
									0, 0,                                    # shared mem and stream
									kernelArgs, 0))   
checkCudaErrors(cudart.cudaDeviceSynchronize())
checkCudaErrors(cuda.cuLaunchKernel(_cudaContourFillerYdiag,
									dimGridy.x, dimGridy.y, dimGridy.z,         # grid dim
									dimBlock.x, dimBlock.y, dimBlock.z,      # block dim
									0, 0,                                    # shared mem and stream
									kernelArgs, 0))  
checkCudaErrors(cudart.cudaDeviceSynchronize())

kernelArgs = ((d_vol, 1, 1, cntr_arraySize[0], cntr_arraySize[1], cntr_arraySize[2], tex), (ctypes.c_void_p, ctypes.c_int, ctypes.c_int, ctypes.c_int, ctypes.c_int, ctypes.c_int, None))
checkCudaErrors(cuda.cuLaunchKernel(_cudaContourFillerXdiag,
									dimGridx.x, dimGridx.y, dimGridx.z,         # grid dim
									dimBlock.x, dimBlock.y, dimBlock.z,      # block dim
									0, 0,                                    # shared mem and stream
									kernelArgs, 0))   
checkCudaErrors(cudart.cudaDeviceSynchronize())
checkCudaErrors(cuda.cuLaunchKernel(_cudaContourFillerYdiag,
									dimGridy.x, dimGridy.y, dimGridy.z,         # grid dim
									dimBlock.x, dimBlock.y, dimBlock.z,      # block dim
									0, 0,                                    # shared mem and stream
									kernelArgs, 0))  
checkCudaErrors(cudart.cudaDeviceSynchronize())

kernelArgs = ((d_vol, 1, -1, cntr_arraySize[0], cntr_arraySize[1], cntr_arraySize[2], tex), (ctypes.c_void_p, ctypes.c_int, ctypes.c_int, ctypes.c_int, ctypes.c_int, ctypes.c_int, None))
checkCudaErrors(cuda.cuLaunchKernel(_cudaContourFillerXdiag,
									dimGridx.x, dimGridx.y, dimGridx.z,         # grid dim
									dimBlock.x, dimBlock.y, dimBlock.z,      # block dim
									0, 0,                                    # shared mem and stream
									kernelArgs, 0))   
checkCudaErrors(cudart.cudaDeviceSynchronize())
checkCudaErrors(cuda.cuLaunchKernel(_cudaContourFillerYdiag,
									dimGridy.x, dimGridy.y, dimGridy.z,         # grid dim
									dimBlock.x, dimBlock.y, dimBlock.z,      # block dim
									0, 0,                                    # shared mem and stream
									kernelArgs, 0))  
checkCudaErrors(cudart.cudaDeviceSynchronize())

In [ ]:
# h_vol = np.zeros(cntr_arraySize[0]*cntr_arraySize[1]*cntr_arraySize[2], dtype=np.float32)
# checkCudaErrors(cudart.cudaMemcpy(d_vol, cntr_volume, dvol_size, cudart.cudaMemcpyKind.cudaMemcpyHostToDevice))
checkCudaErrors(cudart.cudaMemcpy(cntr_volume, d_vol, dvol_size, cudart.cudaMemcpyKind.cudaMemcpyDeviceToHost))

In [ ]:
c_img = cntr_volume.reshape(cntr_arraySize[2],cntr_arraySize[1],cntr_arraySize[0]).T
px.imshow(c_img[:,:,int(0.5*cntr_arraySize[2])-2])

In [ ]:
checkCudaErrors(cudart.cudaDestroyTextureObject(tex))
checkCudaErrors(cudart.cudaFree(d_vol))
checkCudaErrors(cudart.cudaFreeArray(cu_3darray))

In [ ]:
modifiedBresenham(roidf, cntr_volume, cntr_arraySize, value=11.0)

In [ ]:
# send the contour volume, containing the subcontour outlines, to texture memory on the GPU
# /////////////////// Bind Inputs to 3D Texture Arrays //////////////////////////////////////////////
cExtent = cudart.make_cudaExtent(cntr_arraySize[0], cntr_arraySize[1], cntr_arraySize[2])
channelDesc	= checkCudaErrors(cudart.cudaCreateChannelDesc(32, 0, 0, 0, cudart.cudaChannelFormatKind.cudaChannelFormatKindFloat))
cu_3darray =  checkCudaErrors(cudart.cudaMalloc3DArray(channelDesc, cExtent, cudart.cudaArrayDefault))

copyParams = cudart.cudaMemcpy3DParms()
copyParams.srcPos 		= 	cudart.make_cudaPos(0,0,0)
copyParams.dstPos 		= 	cudart.make_cudaPos(0,0,0)
copyParams.srcPtr 		=	cudart.make_cudaPitchedPtr(cntr_volume, cExtent.width*np.dtype(np.float32).itemsize, cExtent.width, cExtent.height);
copyParams.dstArray		= 	cu_3darray
copyParams.extent       =	cExtent
copyParams.kind		    =	cudart.cudaMemcpyKind.cudaMemcpyHostToDevice
checkCudaErrors(cudart.cudaMemcpy3D(copyParams))

texRes = cudart.cudaResourceDesc()
texRes.resType = cudart.cudaResourceType.cudaResourceTypeArray
texRes.res.array.array = cu_3darray

texCntr = cudart.cudaTextureDesc()
texCntr.normalizedCoords	=	False
texCntr.filterMode			=	cudart.cudaTextureFilterMode.cudaFilterModeLinear
texCntr.addressMode[0]		=	cudart.cudaTextureAddressMode.cudaAddressModeClamp
texCntr.addressMode[1]		=	cudart.cudaTextureAddressMode.cudaAddressModeClamp
texCntr.addressMode[2]		=	cudart.cudaTextureAddressMode.cudaAddressModeClamp
texCntr.readMode = cudart.cudaTextureReadMode.cudaReadModeElementType

tex = checkCudaErrors(cudart.cudaCreateTextureObject(texRes, texCntr, None))

In [ ]:
iso_resolution = 1 # min(cntr_resolution)
print(iso_resolution)
res_ratios = [iso_resolution / cntr_resolution[0], iso_resolution / cntr_resolution[1], iso_resolution / cntr_resolution[2]]
print(res_ratios)
iso_arraySize = np.array([int(cntr_arraySize[0]/res_ratios[0] + 0.5), int(cntr_arraySize[1]/res_ratios[1] + 0.5), int(cntr_arraySize[2]/res_ratios[2] + 0.5)])
print(iso_arraySize)


1
[3.1665611146295123, 3.1665611146295123, 0.45310376076121434]
[20 17 29]


In [ ]:
dimBlock.x = 32
dimBlock.y = 32
dimBlock.z = 1

gridx = iso_arraySize[0]/dimBlock.x
if ( iso_arraySize[0] % dimBlock.x > 0): gridx += 1
gridy = iso_arraySize[1]/dimBlock.y
if ( iso_arraySize[1] % dimBlock.y > 0): gridy += 1
gridz = iso_arraySize[2]
	
dimGrid = cudart.dim3()
dimGrid.x = gridx
dimGrid.y = gridy
dimGrid.z = gridz

print(dimBlock)
print(dimGrid)

x : 32
y : 32
z : 1
x : 1
y : 1
z : 29


In [ ]:
ivol_size = iso_arraySize[0] * iso_arraySize[1] * iso_arraySize[2] * np.dtype(np.float32).itemsize
d_vol = checkCudaErrors(cudart.cudaMalloc(ivol_size))
checkCudaErrors(cudart.cudaMemset(d_vol, 0, ivol_size))

In [ ]:
kernelArgs = ((d_vol, iso_arraySize[0], iso_arraySize[1], iso_arraySize[2], res_ratios[0], res_ratios[1], res_ratios[2], 10., tex), 
			  (ctypes.c_void_p, ctypes.c_int, ctypes.c_int, ctypes.c_int, ctypes.c_float, ctypes.c_float, ctypes.c_float, ctypes.c_float, None))
checkCudaErrors(cuda.cuLaunchKernel(_cudaZeroFillerIsotropic,
									dimGrid.x, dimGrid.y, dimGrid.z,         # grid dim
									dimBlock.x, dimBlock.y, dimBlock.z,      # block dim
									0, 0,                                    # shared mem and stream
									kernelArgs, 0))   
checkCudaErrors(cudart.cudaDeviceSynchronize())

In [ ]:
iso_volume = np.zeros(iso_arraySize[0]*iso_arraySize[1]*iso_arraySize[2], dtype=np.float32)

In [ ]:
checkCudaErrors(cudart.cudaMemcpy(iso_volume, d_vol, ivol_size, cudart.cudaMemcpyKind.cudaMemcpyDeviceToHost))

In [ ]:
i_img = iso_volume.reshape(iso_arraySize[2],iso_arraySize[1],iso_arraySize[0]).T
#temp = i_img.T.reshape(-1)
#print(np.array_equal(iso_volume,temp))
px.imshow(i_img[:,:,int(0.5*iso_arraySize[2])])

In [ ]:
checkCudaErrors(cudart.cudaDestroyTextureObject(tex))
checkCudaErrors(cudart.cudaFree(d_vol))
checkCudaErrors(cudart.cudaFreeArray(cu_3darray))

In [ ]:
import cudaSetConnect
connector = cudaSetConnect.SetConnect(iso_arraySize, iso_resolution, iso_volume)
connector.findConnections()

CUDA device [b'NVIDIA GeForce RTX 4090'] has 128 Multi-Processors SM 8.9
[0. 0. 0.]  World Origin
[20 17 29]  Grid Dimensions
[1. 1. 1.]  Resolution
156  Total Particles/Vertices
155
2648  Total Springs


In [ ]:
connector.meshifyVolume()

***  402  Tetrahedrons
***  232  Surface Triangles
***  0.08  cubic cm Total Volume


In [ ]:
connector.vizLineSet()

In [ ]:
connector.vizVolumePoints()

In [ ]:
connector.vizSurfaceTriMesh(w_points=True)

In [ ]:
connector.vizVolumeTetMesh(w_points=True)

In [ ]:
# import cudaSetConnect
# import warp as wp
# import warp.optim
# import warp.sim
# import warp.sim.render

In [ ]:
# import warp as wp
# import warp.sim
# import warp.sim.render

In [ ]:
# warp.init()

In [ ]:
train_iters=10
num_frames=100
# stage_path='usd_files/stages/proto_cage.usd'

In [65]:
import springStiffnessOptimizer

In [66]:
proto = springStiffnessOptimizer.ProtoCage(connector,
                                           num_frames=num_frames,
                                           train_iters=train_iters)

10001


ErrorException: 
	Error in 'pxrInternal_v0_25_5__pxrReserved__::SdfLayer::_CreateNew' at line 599 in file /opt/USD/pxr/usd/sdf/layer.cpp : 'A layer already exists with identifier '/home/jacko/Code/warpbiomechanics/usd_files/stages/cage_proto.usd''

In [59]:
loss = 99999

for iteration in range(train_iters):
# iteration = 0
# while loss > 0.005:
    proto.step()
    loss = proto.loss.numpy()
    print(f"[{iteration:3d}] loss={loss.mean():.8f}")

    if proto.renderer:# and (iteration == 0 or iteration == (train_iters-1)):
    # (iteration == train_iters - 1 or iteration % proto.render_iteration_steps == 0):
        proto.render()

    iteration += 1

if proto.renderer:
    proto.renderer.save()

    Module springStiffnessOptimizer 968874a load on device 'cuda:0' took 219.00 ms  (compiled)
    Module warp.sim.integrator_euler 604c2a7 load on device 'cuda:0' took 0.40 ms  (cached)
    Module warp.sim.particles fe53145 load on device 'cuda:0' took 0.17 ms  (cached)
    Module warp.sim.integrator 3b115ab load on device 'cuda:0' took 0.17 ms  (cached)
Spring stiffness: [10.050917 10.16022  10.62474  ... 10.008495  9.995601  9.993387]
step took 3143.69 ms
[  0] loss=4.99211168
render took 2630.69 ms
Spring stiffness: [10.102825  10.324368  11.25448   ... 10.0169525  9.991218   9.986768 ]
step took 2694.67 ms
[  1] loss=9.98371983
render took 2661.19 ms
Spring stiffness: [10.155873 10.493472 11.891213 ... 10.025369  9.986853  9.980142]
step took 2799.24 ms
[  2] loss=14.97481632
render took 2546.37 ms
Spring stiffness: [10.210238  10.6687565 12.537203  ... 10.033741   9.982506   9.97351  ]
step took 2824.61 ms
[  3] loss=19.96539116
render took 2554.11 ms
Spring stiffness: [10.266132

In [60]:
# spring_ks = proto.model.spring_stiffness.numpy()
# spring_idx = proto.model.spring_indices.numpy().reshape(-1,2)

In [61]:
avg_stiffness = proto.get_average_stiffness_per_particles()

# Map average stiffness to 3D volume
stiffness_map = np.zeros_like(connector.index_volume, dtype=np.float32)
for i, idx in enumerate(connector.nzindices):
    stiffness_map[idx] = avg_stiffness[i]

In [62]:
k_img = stiffness_map.reshape(iso_arraySize[2],iso_arraySize[1],iso_arraySize[0]).T
#temp = i_img.T.reshape(-1)
#print(np.array_equal(iso_volume,temp))
px.imshow(k_img[:,:,int(0.5*iso_arraySize[2])])

In [ ]:
# @wp.kernel
# def assign_param(params: wp.array(dtype=wp.float32), tet_materials: wp.array2d(dtype=wp.float32)):
#     tid = wp.tid()
#     params_idx = 2 * wp.tid() % params.shape[0]
#     tet_materials[tid, 0] = params[params_idx]
#     tet_materials[tid, 1] = params[params_idx + 1]

In [ ]:
# @wp.kernel
# def com_kernel(particle_q: wp.array(dtype=wp.vec3), com: wp.array(dtype=wp.vec3)):
#     tid = wp.tid()
#     point = particle_q[tid]
#     a = point / wp.float32(particle_q.shape[0])

#     # Atomically add the point coordinates to the accumulator
#     wp.atomic_add(com, 0, a)

In [ ]:
# @wp.kernel
# def loss_kernel(
#     target: wp.vec3,
#     com: wp.array(dtype=wp.vec3),
#     pos_error: wp.array(dtype=float),
#     loss: wp.array(dtype=float),
# ):
#     diff = com[0] - target
#     pos_error[0] = wp.dot(diff, diff)
#     norm = pos_error[0]
#     loss[0] = norm

# @wp.kernel
# def energy_loss_kernel(
#     target: wp.vec3,
#     com: wp.array(dtype=wp.vec3),
#     force: wp.array(dtype=float),
#     loss: wp.array(dtype=float),
# ):
#     diff = com[0] - target
#     pos_error[0] = wp.dot(diff, diff)
#     norm = pos_error[0]
#     loss[0] = norm

In [ ]:
# @wp.kernel
# def enforce_constraint_kernel(lower_bound: wp.float32, upper_bound: wp.float32, x: wp.array(dtype=wp.float32)):
#     tid = wp.tid()
#     if x[tid] < lower_bound:
#         x[tid] = lower_bound
#     elif x[tid] > upper_bound:
#         x[tid] = upper_bound

In [ ]:
# class InvElastic:
#     def __init__(
#         self,
#         stage_path="elastic_softbody_properties.usd",
#         material_behavior="anisotropic",
#         verbose=True,
#         volume=None,
#         arraySize=np.array([2,2,2]),
#         resolution=1.0,
#         du=None,
#         dv=None,
#         dw=None
#     ):
#         self.verbose = verbose
#         self.material_behavior = material_behavior
#         self.binary_volume = volume
#         self.binary_3d = volume.reshape(arraySize[2],arraySize[1],arraySize[0]).T
#         self.du = du
#         self.dv = dv
#         self.dw = dw

#         # seconds
#         sim_duration = 1.0

#         # control frequency
#         fps = 60
#         self.frame_dt = 1.0 / fps
#         frame_steps = int(sim_duration / self.frame_dt)

#         # sim frequency
#         self.sim_substeps = 16
#         self.sim_steps = frame_steps * self.sim_substeps
#         self.sim_dt = self.frame_dt / self.sim_substeps

#         self.iter = 0
#         self.render_time = 0.0

#         self.train_rate = 1e7

#         self.losses = []

#         self.hard_lower_bound = wp.float32(500.0)
#         self.hard_upper_bound = wp.float32(4e6)

#         # Create FEM model.
#         if arraySize is None:
#             self.cell_dim = 2
#         else:
#             self.arraySize = arraySize
#             self.cell_dim = arraySize[2]
#         self.cell_size = resolution / 1000. # m
#         # center = self.cell_size * self.cell_dim * 0.5
#         self.grid_origin = wp.vec3(0, 0, 0)
#         self.create_model()

#         self.integrator = wp.sim.SemiImplicitIntegrator()

#         self.target = 0 # (u,v,w) + pos
#         # self.target = wp.vec3(-1.0, 1.5, 0.0)

#         # Initialize material parameters
#         if self.material_behavior == "anisotropic":
#             # Different Lame parameters for each tet
#             self.material_params = wp.array(
#                 self.model.tet_materials.numpy()[:, :2].flatten(),
#                 dtype=wp.float32,
#                 requires_grad=True,
#             )
#         else:
#             # Same Lame parameters for all tets
#             self.material_params = wp.array(
#                 self.model.tet_materials.numpy()[0, :2].flatten(),
#                 dtype=wp.float32,
#                 requires_grad=True,
#             )

#         self.optimizer = wp.optim.SGD(
#             [self.material_params],
#             lr=self.train_rate,
#             nesterov=False,
#         )

#         self.com = wp.array([wp.vec3(0.0, 0.0, 0.0)], dtype=wp.vec3, requires_grad=True)
#         self.pos_error = wp.zeros(1, dtype=wp.float32, requires_grad=True)
#         self.loss = wp.zeros(1, dtype=wp.float32, requires_grad=True)

#         # allocate sim states for trajectory
#         self.states = []
#         for _i in range(self.sim_steps + 1):
#             self.states.append(self.model.state())

#         if stage_path:
#             self.renderer = wp.sim.render.SimRenderer(self.model, stage_path, scaling=1.0)
#         else:
#             self.renderer = None

#         # capture forward/backward passes
#         self.use_cuda_graph = wp.get_device().is_cuda
#         if self.use_cuda_graph:
#             with wp.ScopedCapture() as capture:
#                 self.tape = wp.Tape()
#                 with self.tape:
#                     self.forward()
#                 self.tape.backward(self.loss)
#             self.graph = capture.graph

#     def create_model(self):
#         builder = wp.sim.ModelBuilder()
#         builder.default_particle_radius = 0.001

#         connector = cudaSetConnect(self.arraySize, self.cell_size, self.binary_volume)
#         connector.findConnections()
#         connector.meshifyVolume()

#         # num_particles = connector.numParticles
#         density = 250.0 # kg / m**3 - approximate density of lung
#         particle_mass = density * self.cell_size ** 3
#         # total_mass = particle_mass * num_particles
#         particle_density = particle_mass / (self.cell_size**3)
#         if self.verbose:
#             print(f"Particle density: {particle_density}")

#         young_mod = 5000.0
#         poisson_ratio = 0.43
#         k_mu = 0.5 * young_mod / (1.0 + poisson_ratio)
#         k_lambda = young_mod * poisson_ratio / ((1 + poisson_ratio) * (1 - 2 * poisson_ratio))

#         builder.add_soft_mesh(
#             pos=self.grid_origin,
#             rot=wp.quat_identity(),
#             scale=1.0,
#             vel=wp.vec3(0.0, 0.0, 0.0),
#             vertices=connector.getVolumePoints().tolist(),
#             indices=connector.tetras.flatten().tolist(),
#             density=particle_density,
#             k_mu=k_mu,
#             k_lambda=k_lambda,
#             k_damp=0.0,
#             tri_ke=1e-4,
#             tri_ka=1e-4,
#             tri_kd=1e-4,
#             tri_drag=0.0,
#             tri_lift=0.0,
#         )
#         # builder.add_soft_body(
#         #     pos=self.grid_origin,
#         #     rot=wp.quat_identity(),
#         #     vel=wp.vec3(0.0, 0.0, 0.0),
#         #     dim_x=self.arraySize[0],
#         #     dim_y=self.arraySize[1],
#         #     dim_z=self.arraySize[2],
#         #     cell_x=self.cell_size,
#         #     cell_y=self.cell_size,
#         #     cell_z=self.cell_size,
#         #     density=particle_density,
#         #     k_mu=k_mu,
#         #     k_lambda=k_lambda,
#         #     k_damp=0.0,
#         #     tri_ke=1e-4,
#         #     tri_ka=1e-4,
#         #     tri_kd=1e-4,
#         #     tri_drag=0.0,
#         #     tri_lift=0.0,
#         #     fix_bottom=False,
#         # )
#         ke = 1.0e3
#         kf = 0.0
#         kd = 1.0e0
#         mu = 0.2
#         # builder.add_shape_box(
#         #     body=-1,
#         #     pos=wp.vec3(2.0, 1.0, 0.0),
#         #     hx=0.25,
#         #     hy=1.0,
#         #     hz=1.0,
#         #     ke=ke,
#         #     kf=kf,
#         #     kd=kd,
#         #     mu=mu,
#         # )

#         # use `requires_grad=True` to create a model for differentiable simulation
#         self.model = builder.finalize(requires_grad=True)
#         self.model.ground = True

#         self.model.soft_contact_ke = ke
#         self.model.soft_contact_kf = kf
#         self.model.soft_contact_kd = kd
#         self.model.soft_contact_mu = mu
#         self.model.soft_contact_margin = 0.001
#         self.model.soft_contact_restitution = 1.0

#     def forward(self):
#         wp.launch(
#             kernel=assign_param,
#             dim=self.model.tet_count,
#             inputs=(self.material_params,),
#             outputs=(self.model.tet_materials,),
#         )
#         # run control loop
#         for i in range(self.sim_steps):
#             wp.sim.collide(self.model, self.states[i])
#             self.states[i].clear_forces()

#             self.integrator.simulate(self.model, self.states[i], self.states[i + 1], self.sim_dt)

#         # Update loss
#         # Compute the center of mass for the last time step.
#         wp.launch(
#             kernel=com_kernel,
#             dim=self.model.particle_count,
#             inputs=(self.states[-1].particle_q,),
#             outputs=(self.com,),
#         )

#         # calculate loss
#         wp.launch(
#             kernel=loss_kernel,
#             dim=1,
#             inputs=(
#                 self.target,
#                 self.com,
#             ),
#             outputs=(self.pos_error, self.loss),
#         )

#         return self.loss

#     def step(self):
#         with wp.ScopedTimer("step"):
#             if self.use_cuda_graph:
#                 wp.capture_launch(self.graph)
#             else:
#                 self.tape = wp.Tape()
#                 with self.tape:
#                     self.forward()
#                 self.tape.backward(loss=self.loss)

#             if self.verbose:
#                 self.log_step()

#             self.optimizer.step([self.material_params.grad])

#             wp.launch(
#                 kernel=enforce_constraint_kernel,
#                 dim=self.material_params.shape[0],
#                 inputs=(
#                     self.hard_lower_bound,
#                     self.hard_upper_bound,
#                 ),
#                 outputs=(self.material_params,),
#             )

#             self.losses.append(self.loss.numpy()[0])

#             # clear grads for next iteration
#             self.tape.zero()
#             self.loss.zero_()
#             self.com.zero_()
#             self.pos_error.zero_()

#             self.iter = self.iter + 1

#     def log_step(self):
#         x = self.material_params.numpy().reshape(-1, 2)
#         x_grad = self.material_params.grad.numpy().reshape(-1, 2)

#         print(f"Iter: {self.iter} Loss: {self.loss.numpy()[0]}")

#         print(f"Pos error: {np.sqrt(self.pos_error.numpy()[0])}")

#         print(
#             f"Max Mu: {np.max(x[:, 0])}, Min Mu: {np.min(x[:, 0])}, "
#             f"Max Lambda: {np.max(x[:, 1])}, Min Lambda: {np.min(x[:, 1])}"
#         )

#         print(
#             f"Max Mu Grad: {np.max(x_grad[:, 0])}, Min Mu Grad: {np.min(x_grad[:, 0])}, "
#             f"Max Lambda Grad: {np.max(x_grad[:, 1])}, Min Lambda Grad: {np.min(x_grad[:, 1])}"
#         )

#     def render(self):
#         if self.renderer is None:
#             return

#         with wp.ScopedTimer("render"):
#             # draw trajectory
#             traj_verts = [np.mean(self.states[0].particle_q.numpy(), axis=0).tolist()]
#             for i in range(0, self.sim_steps, self.sim_substeps):
#                 traj_verts.append(np.mean(self.states[i].particle_q.numpy(), axis=0).tolist())

#                 self.renderer.begin_frame(self.render_time)
#                 self.renderer.render(self.states[i])
#                 self.renderer.render_box(
#                     pos=self.target,
#                     rot=wp.quat_identity(),
#                     extents=(0.1, 0.1, 0.1),
#                     name="target",
#                     color=(0.0, 0.0, 0.0),
#                 )
#                 self.renderer.render_line_strip(
#                     vertices=traj_verts,
#                     color=wp.render.bourke_color_map(0.0, self.losses[0], self.losses[-1]),
#                     radius=0.02,
#                     name=f"traj_{self.iter - 1}",
#                 )
#                 self.renderer.end_frame()

#                 from pxr import Gf, UsdGeom

#                 particles_prim = self.renderer.stage.GetPrimAtPath("/root/particles")
#                 particles = UsdGeom.Points.Get(self.renderer.stage, particles_prim.GetPath())
#                 particles.CreateDisplayColorAttr().Set([Gf.Vec3f(1.0, 1.0, 1.0)], time=self.renderer.time)

#                 self.render_time += self.frame_dt

In [ ]:
# sdf = connector.getSDF()

In [ ]:
# sdf = sdf.reshape(connector.arraySize[2],connector.arraySize[1],connector.arraySize[0]).T

In [ ]:
# sdf.shape

In [ ]:
# px.imshow(sdf[:,:,int(0.5*iso_arraySize[2])+10])

In [ ]:
# wp.init()

In [ ]:
# @wp.kernel
# def compute_volume(points: wp.array(dtype=wp.vec3), indices: wp.array2d(dtype=int), volume: wp.array(dtype=float)):
#     tid = wp.tid()

#     i = indices[tid, 0]
#     j = indices[tid, 1]
#     k = indices[tid, 2]
#     l = indices[tid, 3]

#     x0 = points[i]
#     x1 = points[j]
#     x2 = points[k]
#     x3 = points[l]

#     x10 = x1 - x0
#     x20 = x2 - x0
#     x30 = x3 - x0

#     v = wp.dot(x10, wp.cross(x20, x30)) / 6.0

#     wp.atomic_add(volume, 0, v)

In [ ]:
# @wp.kernel
# def twist_points(
#     rest: wp.array(dtype=wp.vec3), points: wp.array(dtype=wp.vec3), mass: wp.array(dtype=float), xform: wp.transform
# ):
#     tid = wp.tid()

#     r = rest[tid]
#     p = points[tid]
#     m = mass[tid]

#     # twist the top layer of particles in the beam
#     if m == 0 and p[1] != 0.0:
#         points[tid] = wp.transform_point(xform, r)

In [ ]:
# @wp.kernel
# def deform(positions: wp.array(dtype=wp.vec3), t: float):
#     tid = wp.tid()

#     x = positions[tid]

#     offset = -wp.sin(x[0]) * 0.02
#     scale = wp.sin(t)

#     x = x + wp.vec3(0.0, offset * scale, 0.0)

#     positions[tid] = x

In [ ]:
# @wp.kernel
# def simulate(
#     positions: wp.array(dtype=wp.vec3),
#     velocities: wp.array(dtype=wp.vec3),
#     mesh: wp.uint64,
#     margin: float,
#     dt: float,
# ):
#     tid = wp.tid()

#     x = positions[tid]
#     v = velocities[tid]

#     v = v + wp.vec3(0.0, 0.0 - 9.8, 0.0) * dt - v * 0.1 * dt
#     xpred = x + v * dt

#     max_dist = 1.5

#     query = wp.mesh_query_point_sign_normal(mesh, xpred, max_dist)
#     if query.result:
#         p = wp.mesh_eval_position(mesh, query.face, query.u, query.v)

#         delta = xpred - p

#         dist = wp.length(delta) * query.sign
#         err = dist - margin

#         # mesh collision
#         if err < 0.0:
#             n = wp.normalize(delta) * query.sign
#             xpred = xpred - n * err

#     # pbd update
#     v = (xpred - x) * (1.0 / dt)
#     x = xpred

#     positions[tid] = x
#     velocities[tid] = v

In [ ]:
# class TwistSoftBody:
#     def __init__(self, usd_path="/home/jacko/Code/dynamo/usd_files/rois", model_name="Eye_L", num_frames=300):
#         rng = np.random.default_rng(42)
#         self.stage_path = usd_path + model_name + '-render.usda' 
#         self.model_path = usd_path + model_name + '.usda'
#         # self.sim_substeps = 64
#         self.num_particles = 1000
#         self.num_frames = num_frames
#         fps = 60
#         sim_duration = self.num_frames / fps
#         # self.frame_dt = 1.0 / fps
#         # self.sim_dt = self.frame_dt / self.sim_substeps
#         self.sim_dt = 1.0 / fps
#         self.sim_time = 0.0
#         self.sim_timers = {}
#         self.sim_margin = 0.1
#         # self.lift_speed = 2.5 / sim_duration * 2.0  # from Smith et al.
#         # self.rot_speed = math.pi / sim_duration

#         usd_stage = Usd.Stage.Open(self.model_path)
#         usd_geom = UsdGeom.Mesh(usd_stage.GetPrimAtPath('/Root/primitive'))
#         usd_scale = 1.0

#         self.mesh = wp.Mesh(
#             points=wp.array(usd_geom.GetPointsAttr().Get() * usd_scale, dtype=wp.vec3),
#             indices=wp.array(usd_geom.GetFaceVertexIndicesAttr().Get(), dtype=int),
#         )

#         # random particles
#         init_pos = (rng.random((self.num_particles, 3)) - np.array([0.5, -1.5, 0.5])) * 50.0
#         init_vel = rng.random((self.num_particles, 3)) * 0.0

#         self.positions = wp.from_numpy(init_pos, dtype=wp.vec3)
#         self.velocities = wp.from_numpy(init_vel, dtype=wp.vec3)

#         # builder = wp.sim.ModelBuilder()
#         # cell_dim = 15
#         # cell_size = 2.0 / cell_dim
#         # center = cell_size * cell_dim * 0.5
#         # builder.add_soft_grid(
#         #     pos=wp.vec3(-center, 0.0, -center),
#         #     rot=wp.quat_identity(),
#         #     vel=wp.vec3(0.0, 0.0, 0.0),
#         #     dim_x=cell_dim,
#         #     dim_y=cell_dim,
#         #     dim_z=cell_dim,
#         #     cell_x=cell_size,
#         #     cell_y=cell_size,
#         #     cell_z=cell_size,
#         #     density=100.0,
#         #     fix_bottom=True,
#         #     fix_top=True,
#         #     k_mu=1000.0,
#         #     k_lambda=5000.0,
#         #     k_damp=0.0,
#         # )
#         # self.model = builder.finalize()
#         # self.model.ground = False
#         # self.model.gravity[1] = 0.0
#         # self.integrator = wp.sim.SemiImplicitIntegrator()
#         # self.rest = self.model.state()
#         # self.rest_vol = (cell_size * cell_dim) ** 3
#         # self.state_0 = self.model.state()
#         # self.state_1 = self.model.state()
#         # self.volume = wp.zeros(1, dtype=wp.float32)

#         if self.stage_path:
#             self.renderer = wp.render.UsdRenderer(self.stage_path)
#             # self.renderer = wp.sim.render.SimRenderer(self.model, stage_path, scaling=20.0)
#         else:
#             self.renderer = None

#         # self.use_cuda_graph = wp.get_device().is_cuda
#         # if self.use_cuda_graph:
#         #     with wp.ScopedCapture() as capture:
#         #         self.twistulate()
#         #     self.graph = capture.graph

#     def twistulate(self):
#         for _ in range(self.sim_substeps):
#             self.state_0.clear_forces()
#             self.state_1.clear_forces()

#             self.integrator.simulate(self.model, self.state_0, self.state_1, self.sim_dt)

#             # swap states
#             (self.state_0, self.state_1) = (self.state_1, self.state_0)

#     def deform_step(self):
#         with wp.ScopedTimer("step", dict=self.sim_timers):
#             wp.launch(kernel=deform, dim=len(self.mesh.points), inputs=[self.mesh.points, self.sim_time])

#             # refit the mesh BVH to account for the deformation
#             self.mesh.refit()

#             wp.launch(
#                 kernel=simulate,
#                 dim=self.num_particles,
#                 inputs=[self.positions, self.velocities, self.mesh.id, self.sim_margin, self.sim_dt],
#             )

#             self.sim_time += self.sim_dt

#     def twist_step(self):
#         with wp.ScopedTimer("step"):
#             xform = wp.transform(
#                 (0.0, self.lift_speed * self.sim_time, 0.0),
#                 wp.quat_from_axis_angle(wp.vec3(0.0, 1.0, 0.0), self.rot_speed * self.sim_time),
#             )
#             wp.launch(
#                 kernel=twist_points,
#                 dim=len(self.state_0.particle_q),
#                 inputs=[self.rest.particle_q, self.state_0.particle_q, self.model.particle_mass, xform],
#             )
#             if self.use_cuda_graph:
#                 wp.capture_launch(self.graph)
#             else:
#                 self.twistulate()
#             self.volume.zero_()
#             wp.launch(
#                 kernel=compute_volume,
#                 dim=self.model.tet_count,
#                 inputs=[self.state_0.particle_q, self.model.tet_indices, self.volume],
#             )
#         self.sim_time += self.frame_dt

#     def render(self):
#         if self.renderer is None:
#             return

#         with wp.ScopedTimer("render"):
#             self.renderer.begin_frame(self.sim_time)
#             # self.renderer.render(self.state_0)
#             self.renderer.render_mesh(
#                 name="mesh",
#                 points=self.mesh.points.numpy(),
#                 indices=self.mesh.indices.numpy(),
#                 colors=(0.35, 0.55, 0.9),
#             )
#             self.renderer.render_points(
#                 name="points", points=self.positions.numpy(), radius=self.sim_margin, colors=(0.8, 0.3, 0.2)
#             )
#             self.renderer.end_frame()

In [ ]:
# usd_path = "usd_files/"
# stage_path = usd_path + "stages/" + 'hilja_leftlung_trimesh.usda' 
# num_frames = 500

In [ ]:
# twister = TwistSoftBody(usd_path=usd_path + "stages/", model_name='hilja_leftlung_trimesh', num_frames=num_frames)

In [ ]:
# for _ in range(num_frames):
#     twister.deform_step()
#     twister.render()

In [ ]:
# if twister.renderer:
#     twister.renderer.save()

In [ ]:
# @wp.kernel
# def eval_tetrahedra(
#     x: wp.array(dtype=wp.vec3),
#     v: wp.array(dtype=wp.vec3),
#     indices: wp.array2d(dtype=int),
#     pose: wp.array(dtype=wp.mat33),
#     activation: wp.array(dtype=float),
#     materials: wp.array2d(dtype=float),
#     f: wp.array(dtype=wp.vec3),
# ):
#     tid = wp.tid()

#     i = indices[tid, 0]
#     j = indices[tid, 1]
#     k = indices[tid, 2]
#     l = indices[tid, 3]

#     act = activation[tid]

#     k_mu = materials[tid, 0]
#     k_lambda = materials[tid, 1]
#     k_damp = materials[tid, 2]

#     x0 = x[i]
#     x1 = x[j]
#     x2 = x[k]
#     x3 = x[l]

#     v0 = v[i]
#     v1 = v[j]
#     v2 = v[k]
#     v3 = v[l]

#     x10 = x1 - x0
#     x20 = x2 - x0
#     x30 = x3 - x0

#     v10 = v1 - v0
#     v20 = v2 - v0
#     v30 = v3 - v0

#     Ds = wp.matrix_from_cols(x10, x20, x30)
#     Dm = pose[tid]

#     inv_rest_volume = wp.determinant(Dm) * 6.0
#     rest_volume = 1.0 / inv_rest_volume

#     alpha = 1.0 + k_mu / k_lambda - k_mu / (4.0 * k_lambda)

#     # scale stiffness coefficients to account for area
#     k_mu = k_mu * rest_volume
#     k_lambda = k_lambda * rest_volume
#     k_damp = k_damp * rest_volume

#     # F = Xs*Xm^-1
#     F = Ds * Dm
#     dFdt = wp.matrix_from_cols(v10, v20, v30) * Dm

#     col1 = wp.vec3(F[0, 0], F[1, 0], F[2, 0])
#     col2 = wp.vec3(F[0, 1], F[1, 1], F[2, 1])
#     col3 = wp.vec3(F[0, 2], F[1, 2], F[2, 2])

#     # -----------------------------
#     # Neo-Hookean (with rest stability [Smith et al 2018])

#     Ic = wp.dot(col1, col1) + wp.dot(col2, col2) + wp.dot(col3, col3)

#     # deviatoric part
#     P = F * k_mu * (1.0 - 1.0 / (Ic + 1.0)) + dFdt * k_damp
#     H = P * wp.transpose(Dm)

#     f1 = wp.vec3(H[0, 0], H[1, 0], H[2, 0])
#     f2 = wp.vec3(H[0, 1], H[1, 1], H[2, 1])
#     f3 = wp.vec3(H[0, 2], H[1, 2], H[2, 2])

#     # -----------------------------
#     # C_sqrt

#     # alpha = 1.0

#     # r_s = wp.sqrt(wp.abs(dot(col1, col1) + dot(col2, col2) + dot(col3, col3) - 3.0))

#     # f1 = wp.vec3()
#     # f2 = wp.vec3()
#     # f3 = wp.vec3()

#     # if (r_s > 0.0):
#     #     r_s_inv = 1.0/r_s

#     #     C = r_s
#     #     dCdx = F*wp.transpose(Dm)*r_s_inv*wp.sign(r_s)

#     #     grad1 = vec3(dCdx[0,0], dCdx[1,0], dCdx[2,0])
#     #     grad2 = vec3(dCdx[0,1], dCdx[1,1], dCdx[2,1])
#     #     grad3 = vec3(dCdx[0,2], dCdx[1,2], dCdx[2,2])

#     #     f1 = grad1*C*k_mu
#     #     f2 = grad2*C*k_mu
#     #     f3 = grad3*C*k_mu

#     # -----------------------------
#     # C_spherical

#     # alpha = 1.0

#     # r_s = wp.sqrt(dot(col1, col1) + dot(col2, col2) + dot(col3, col3))
#     # r_s_inv = 1.0/r_s

#     # C = r_s - wp.sqrt(3.0)
#     # dCdx = F*wp.transpose(Dm)*r_s_inv

#     # grad1 = vec3(dCdx[0,0], dCdx[1,0], dCdx[2,0])
#     # grad2 = vec3(dCdx[0,1], dCdx[1,1], dCdx[2,1])
#     # grad3 = vec3(dCdx[0,2], dCdx[1,2], dCdx[2,2])

#     # f1 = grad1*C*k_mu
#     # f2 = grad2*C*k_mu
#     # f3 = grad3*C*k_mu

#     # ----------------------------
#     # C_D

#     # alpha = 1.0

#     # r_s = wp.sqrt(dot(col1, col1) + dot(col2, col2) + dot(col3, col3))

#     # C = r_s*r_s - 3.0
#     # dCdx = F*wp.transpose(Dm)*2.0

#     # grad1 = vec3(dCdx[0,0], dCdx[1,0], dCdx[2,0])
#     # grad2 = vec3(dCdx[0,1], dCdx[1,1], dCdx[2,1])
#     # grad3 = vec3(dCdx[0,2], dCdx[1,2], dCdx[2,2])

#     # f1 = grad1*C*k_mu
#     # f2 = grad2*C*k_mu
#     # f3 = grad3*C*k_mu

#     # ----------------------------
#     # Hookean

#     # alpha = 1.0

#     # I = wp.matrix_from_cols(wp.vec3(1.0, 0.0, 0.0),
#     #                         wp.vec3(0.0, 1.0, 0.0),
#     #                         wp.vec3(0.0, 0.0, 1.0))

#     # P = (F + wp.transpose(F) + I*(0.0-2.0))*k_mu
#     # H = P * wp.transpose(Dm)

#     # f1 = wp.vec3(H[0, 0], H[1, 0], H[2, 0])
#     # f2 = wp.vec3(H[0, 1], H[1, 1], H[2, 1])
#     # f3 = wp.vec3(H[0, 2], H[1, 2], H[2, 2])

#     # hydrostatic part
#     J = wp.determinant(F)

#     # print(J)
#     s = inv_rest_volume / 6.0
#     dJdx1 = wp.cross(x20, x30) * s
#     dJdx2 = wp.cross(x30, x10) * s
#     dJdx3 = wp.cross(x10, x20) * s

#     f_volume = (J - alpha + act) * k_lambda
#     f_damp = (wp.dot(dJdx1, v1) + wp.dot(dJdx2, v2) + wp.dot(dJdx3, v3)) * k_damp

#     f_total = f_volume + f_damp

#     f1 = f1 + dJdx1 * f_total
#     f2 = f2 + dJdx2 * f_total
#     f3 = f3 + dJdx3 * f_total
#     f0 = -(f1 + f2 + f3)

#     # apply forces
#     wp.atomic_sub(f, i, f0)
#     wp.atomic_sub(f, j, f1)
#     wp.atomic_sub(f, k, f2)
#     wp.atomic_sub(f, l, f3)